In [1]:
import pandas as pd
from presidio_analyzer import AnalyzerEngine
import json
import re

In [2]:
df = pd.DataFrame({
    "col1": [
        "erdenetuya.kovicovi@seznam.cz",
        "vnemec@volny.cz",
        "elena@strnadovi.com",
        "hasmik@sulcovi.eu",
        "lholubovi#gmail&com",
        "ikucerovi@gmail.cz",
        "michele_cechovi@volny.cz",
        "tsebestovi@gmail.cz"
    ],
    "col2": ["Kováčová", "Němec", "Strnadova", "Šulcová", "Holubová", "Kučerová", "Čechova", "Šebestová"],
    "col3": ["Erdenetuya", "Valter", "Elena", "Hasmik", "Luisa", "Isabel", "Michele", "Thanh"]
})

In [3]:
# Vytvoření analyzeru s NLP enginem
analyzer = AnalyzerEngine()

In [4]:
# Identifikace dominantní domény pro každý sloupec
column_domains = {}

for col in df.columns:
    entity_counts = {}
    for val in df[col]:
        entities = analyzer.analyze(text=str(val), language="en", entities=[])
        for ent in entities:
            entity_counts[ent.entity_type] = entity_counts.get(ent.entity_type, 0) + 1

    if entity_counts:
        dominant_entity = max(entity_counts, key=entity_counts.get)
        column_domains[col] = {
            "dominant_entity": dominant_entity,
            "confidence": round(entity_counts[dominant_entity] / len(df[col]), 2)
        }
    else:
        column_domains[col] = {
            "dominant_entity": None,
            "confidence": 0.0
        }

# Výpis domén
print("Zjištěné domény sloupců:")
for col, info in column_domains.items():
    print(f"- {col}: {info['dominant_entity']} ({info['confidence']*100:.0f} %)")

Zjištěné domény sloupců:
- col1: EMAIL_ADDRESS (88 %)
- col2: PERSON (62 %)
- col3: PERSON (75 %)


In [ ]:
# Aplikace pravidla pro e-mail
email_column = None
for col, info in column_domains.items():
    if info["dominant_entity"] == "EMAIL_ADDRESS":
        email_column = col
        break

def is_valid_email(text):
    return bool(re.match(r"^[^@\s]+@[^@\s]+\.[^@\s]+$", str(text)))

invalid_values = []

if email_column:
    for idx, val in enumerate(df[email_column]):
        if not is_valid_email(val):
            invalid_values.append({"index": idx, "value": val})

    validation_result = {
        "column": email_column,
        "entity": "EMAIL_ADDRESS",
        "rule": "valid_email_format",
        "total": len(df),
        "invalid": len(invalid_values),
        "invalid_values": invalid_values
    }

In [6]:
with open("../invalid/export/email_validation.json", "w", encoding="utf-8") as f:
    json.dump(validation_result, f, indent=2, ensure_ascii=False)